# 04 — IEEE-CIS LTN-Style Fraud Rule Analysis

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
KAGGLE = Path("/kaggle").exists()
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
print({"project_root": str(PROJECT_ROOT), "quick_run": QUICK_RUN, "kaggle": KAGGLE})

In [ ]:
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data
from src.explanation import rule_quality_table
from src.logic import FraudKnowledgeBase, FraudRuleEngine

config = load_config(PROJECT_ROOT / "configs/ieee_cis.yaml")
output_dir = OUTPUT_BASE / "03_rule_analysis"
output_dir.mkdir(parents=True, exist_ok=True)
frame, data_source = load_experiment_data(
    config,
    max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
knowledge_base = FraudKnowledgeBase(engine)
print({"data_source": data_source, "active_rules": len(engine.rules), "skipped": engine.skipped_rules})
display(engine.fitted_thresholds())

### Differentiable tensor-logic check

The following diagnostic demonstrates a learnable fuzzy predicate and quantified satisfaction with PyTorch. It is trained on the training split only and is not used to select or alter the main rule-quality results below.

In [ ]:
import torch
from src.logic import SoftThresholdPredicate, TensorLogic

amount = torch.tensor(prepared.train_frame["TransactionAmt"].to_numpy(float), dtype=torch.float32)
amount_center = torch.nanmedian(amount)
amount_scale = torch.nan_to_num(amount.std(), nan=1.0).clamp_min(1e-6)
standardized_amount = torch.nan_to_num((amount - amount_center) / amount_scale)
labels = torch.tensor(prepared.y_train, dtype=torch.float32)
initial_threshold = float(torch.quantile(standardized_amount, 0.90))
tensor_predicate = SoftThresholdPredicate(initial_threshold, temperature=0.5, learnable=True)
optimizer = torch.optim.Adam(tensor_predicate.parameters(), lr=0.03)
initial_parameter = tensor_predicate.threshold.detach().clone()
history = []
for step in range(30 if QUICK_RUN else 100):
    optimizer.zero_grad()
    evidence = tensor_predicate(standardized_amount)
    positive_satisfaction = TensorLogic.forall(evidence[labels == 1])
    negative_satisfaction = TensorLogic.forall(1.0 - evidence[labels == 0])
    satisfaction = 0.5 * (positive_satisfaction + negative_satisfaction)
    regularization = 0.01 * (tensor_predicate.threshold - initial_parameter).pow(2)
    loss = 1.0 - satisfaction + regularization
    loss.backward()
    optimizer.step()
    history.append(float(satisfaction.detach()))

tensor_demo = pd.DataFrame({
    "initial_threshold_standardized": [float(initial_parameter)],
    "learned_threshold_standardized": [float(tensor_predicate.threshold.detach())],
    "initial_satisfaction": [history[0]],
    "final_satisfaction": [history[-1]],
})
display(tensor_demo.round(4))

## Data

In [ ]:
validation_truth = engine.evaluate(prepared.validation_frame)
test_truth = engine.evaluate(prepared.test_frame)
activation_threshold = float(config["logic"]["activation_threshold"])
validation_quality = rule_quality_table(validation_truth, prepared.y_validation, activation_threshold).assign(split="validation")
test_quality = rule_quality_table(test_truth, prepared.y_test, activation_threshold).assign(split="test")
quality = pd.concat([validation_quality, test_quality], ignore_index=True)
display(quality.round(4))

## Results

In [ ]:
stability = validation_quality.merge(test_quality, on="rule", suffixes=("_validation", "_test"))
stability["coverage_delta"] = stability["coverage_test"] - stability["coverage_validation"]
stability["lift_delta"] = stability["lift_test"] - stability["lift_validation"]
display(stability[["rule", "coverage_validation", "coverage_test", "coverage_delta", "lift_validation", "lift_test", "lift_delta"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=quality, x="rule", y="coverage", hue="split", ax=axes[0])
axes[0].set_title("Rule coverage")
axes[0].tick_params(axis="x", rotation=35)
sns.barplot(data=quality, x="rule", y="lift", hue="split", ax=axes[1])
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Fraud lift")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
fig.savefig(output_dir / "rule_quality.png", dpi=160, bbox_inches="tight")
plt.show()

quality.to_csv(output_dir / "rule_quality.csv", index=False)
stability.to_csv(output_dir / "rule_stability.csv", index=False)
engine.fitted_thresholds().to_csv(output_dir / "fitted_rule_thresholds.csv", index=False)

## Takeaways

In [ ]:
strongest = test_quality.sort_values("lift", ascending=False).iloc[0]
display(Markdown(
    f"- Data source: **{data_source}**.\n"
    f"- Active rules: **{len(engine.rules)}**; skipped rules: **{len(engine.skipped_rules)}**.\n"
    f"- Highest test lift: **{strongest['rule']} = {strongest['lift']:.3f}** at coverage **{strongest['coverage']:.3f}**.\n"
    "- Lift indicates association with fraud labels, not causality."
))